# Test Age/Gender Model on a Single Image
This notebook loads your trained `.pth` model and tests it against a single custom image of your choice. You don't need a full dataset for this!

In [ ]:
import torch
from PIL import Image
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Import your model loading function and transform logic
from src.models.MobileNet.runner_scripts.trainer import load_model
from src.models.MobileNet.data_defs import get_transforms

In [ ]:
# 1. Load the model
# UPDATE THIS PATH to wherever your .pth file is stored!
model_path = "model_checkpoint.pth" # Or e.g., "model_store/model_checkpoint.pth"

model = load_model(model_path)
model.eval()  # Set the model to evaluation mode
print("Model loaded successfully!")

In [ ]:
# 2. Load and display an image
# UPDATE THIS PATH to the image you want to test
image_path = "test_image.jpg" 

try:
    # Open the image and force squeeze it to exactly 224x224 (like UTKFace)
    image = Image.open(image_path).convert("RGB")
    image = image.resize((224, 224))
    
    plt.imshow(image)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Could not find image at {image_path}. Please put an image there!")

In [ ]:
# 3. Preprocess the image
# This applies normalization
transform = get_transforms(get_compose=True)
input_tensor = transform(image).unsqueeze(0)  # Add batch dimension

In [ ]:
# 4. Run the prediction
with torch.no_grad():
    gender_logits, age_pred = model(input_tensor)
    
    # Convert gender logits to probabilities
    gender_probs = F.softmax(gender_logits, dim=1)
    predicted_gender_idx = torch.argmax(gender_probs, dim=1).item()
    
    # Assuming UTKFace standard: 0 = Male, 1 = Female
    gender_label = "Female" if predicted_gender_idx == 1 else "Male"
    gender_confidence = gender_probs[0][predicted_gender_idx].item() * 100
    
    predicted_age = age_pred.item()

print("--- Predictions ---")
print(f"Predicted Age: {predicted_age:.1f} years")
print(f"Predicted Gender: {gender_label} ({gender_confidence:.1f}% confidence)")